# train_predict — thin Colab trainer for the `predict` skill

Inputs: a labelled feature table `data/derived/predict_<target>_<asof>.parquet` produced by
`models.predict features` + `models.predict.labels` (validator). Upload it to Colab or
mount Drive. Secrets: `PLANT_API_URL`, `PLANT_ADMIN_TOKEN` in Colab *Secrets* (key icon).

Never commit this notebook with outputs.

In [ ]:
!pip install -q git+https://github.com/RotoPower/learn-cognitive-maintenance

In [ ]:
import pandas as pd
from google.colab import files, userdata
from models.predict import train, save_artifact, upload

HORIZON_DAYS = 30
SEED = 42
CUTOFF = "2024-10-15"  # train before this, replay after; None = 70% point of the table

uploaded = files.upload()  # pick predict_<target>_<asof>.parquet
table = pd.read_parquet(next(iter(uploaded)))
assert "label" in table.columns, "table has no label column: run models.predict.labels first"
table.shape, table["label"].value_counts(dropna=False).to_dict()

In [ ]:
art = train(table, horizon_days=HORIZON_DAYS, seed=SEED, cutoff=CUTOFF)
path = save_artifact(art)
te = art["metrics"]["test"] or {}
print(art["run_id"])
print({k: te.get(k) for k in ("pr_auc", "precision_at_5", "lead_time_days_mean", "false_alerts_per_asset_month")})

In [ ]:
# Upload if the plant API is reachable from Colab (staging URL, Part D); otherwise download the
# artefact and upload it locally:  uv run plantctl --admin upload-artifact --file models/artifacts/<run_id>.json
try:
    api_url = userdata.get("PLANT_API_URL")
    token = userdata.get("PLANT_ADMIN_TOKEN")
    print(upload(art, api_url, token))
except Exception as e:  # secret missing or API not reachable from Colab
    print(f"upload skipped ({type(e).__name__}: {e}); download the artefact and upload it locally with plantctl")
files.download(str(path))  # always keep a local copy: models/artifacts/<run_id>.json
